<a href="https://colab.research.google.com/github/pranav-deshpande6224/DDPM_DIT_DDIM/blob/main/faces_gen_DDPM_DIT_DDIM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# This project is mainly about
# Latent Diffusion Model with UNET VS DIT[Diffusion Transformer] VS DDIM


**IN DIT instead of UNET we use the Transformer for Denoising the image**

* Train the VQVAE On Celeb Faces HQ dataset and save the checkpoint


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import os
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import torch.optim as optim
from tqdm import tqdm
import torchvision.utils as vutils
import torchvision
from torch.nn.modules.activation import SiLU

# VQVAE
* Encoder Down
* Mid Block
* UpBlock

In [ ]:
class MHAttention(nn.Module):
  def __init__(self,channels_group,in_channel,num_heads):
    super().__init__()
    self.channels_group = channels_group
    self.in_channel = in_channel
    self.num_heads = num_heads
    self.norm = nn.GroupNorm(self.channels_group, in_channel)
    self.attention= nn.MultiheadAttention(self.in_channel, num_heads=self.num_heads,batch_first=True)

  def forward(self, x):
    input = x
    x = self.norm(x)
    bs,ch,h,w = x.shape
    x = x.reshape(bs, ch, h*w)
    x = x.transpose(1,2)
    x, _ = self.attention(x,x,x)
    x = x.transpose(1,2).reshape(bs,ch,h,w)
    out = input + x
    return out

class ResidualBlock(nn.Module):
  def __init__(self, in_channel, out_channel,channels_group,t_emb_info = None):
    super().__init__()
    self.channels_group = channels_group
    self.in_channel = in_channel
    self.out_channel = out_channel
    self.t_emb_info = t_emb_info
    self.res_1 = nn.Sequential(
            nn.GroupNorm(self.channels_group, self.in_channel),
            nn.SiLU(),
            nn.Conv2d(in_channels=self.in_channel,out_channels=self.out_channel,kernel_size=3,stride=1,padding=1),

          )
    if t_emb_info is not None:
      self.t_emb_layers = nn.Sequential(
              nn.SiLU(),
              nn.Linear(t_emb_info, self.out_channel)
          )
    self.res_2 = nn.Sequential(
            nn.GroupNorm(self.channels_group, self.out_channel),
            nn.SiLU(),
            nn.Conv2d(in_channels=self.out_channel,out_channels=self.out_channel,kernel_size=3,stride=1,padding=1),

          )
    self.residual_conn = nn.Conv2d(in_channels=self.in_channel,out_channels=self.out_channel,kernel_size=1,stride=1) if self.in_channel != self.out_channel else nn.Identity()

  def forward(self,x,t_emb = None):
      input = x
      x = self.res_1(x)
      if self.t_emb_info:
        x = x + self.t_emb_layers(t_emb)[:,:,None,None]
      x = self.res_2(x)
      x = x + self.residual_conn(input)
      return x

In [ ]:
class Encoder_Down(nn.Module):
  def __init__(self, in_channel, out_channel, channel_group,n_layers):
    super().__init__()
    self.n_layers = n_layers
    self.residual = nn.ModuleList([
        ResidualBlock(in_channel if i == 0 else out_channel, out_channel,channel_group)
        for i in range(n_layers)
    ])
    self.downsample = nn.Conv2d(in_channels=out_channel, out_channels= out_channel,kernel_size=4,stride=2,padding=1)
  def forward(self, x):
    out = x
    for i in range(self.n_layers):
      out = self.residual[i](out)

    out = self.downsample(out)
    return out

In [ ]:
class MidBlock(nn.Module):
  def __init__(self,in_channel, out_channel, channel_group,n_layers,n_heads):
    super().__init__()
    self.n_layers = n_layers
    self.residual_first = nn.ModuleList([
        ResidualBlock(in_channel if i == 0 else out_channel, out_channel,channel_group)
        for i in range(n_layers + 1)
    ])
    self.mha = nn.ModuleList([
        MHAttention(channel_group, out_channel,n_heads)
        for i in range(n_layers)
    ])
  def forward(self, x):

    out = x
    resnet_input = out
    out = self.residual_first[0](out)
    for i in range(self.n_layers):
      out = self.mha[i](out)
      resnet_input = out
      out = self.residual_first[i+1](out)

    return out

In [ ]:
class UpBlock(nn.Module):
  def __init__(self,in_channel, out_channel, channel_group,n_layers):
    super().__init__()
    self.n_layers = n_layers
    self.residual = nn.ModuleList([
        ResidualBlock(in_channel if i == 0 else out_channel, out_channel,channel_group)
        for i in range(n_layers)
    ])
    self.upsample_conv = nn.ConvTranspose2d(in_channel, in_channel, 4, 2, 1)
  def forward(self,x):
    x = self.upsample_conv(x)
    for i in range(self.n_layers):
      x = self.residual[i](x)

    return x

In [ ]:
class VQVAE(nn.Module):
  def __init__(self):

    super().__init__()
    self.image_channels = 3
    self.downsample_channels = [64,128,256,256]
    self.mid_channels = [256,256]
    self.num_down_layers = 2
    self.mid_num_layers = 2
    self.num_up_layers = 2
    self.group_norm_channels = 32
    self.num_heads = 4

    #After the mid block
    self.z_channels = 3
    self.codebook_size = 8192
    self.upsample_channels = self.downsample_channels[::-1]

    #before the encoder convert RGB to downsample_channels[0]
    self.before_encoder = nn.Conv2d(in_channels=self.image_channels,out_channels=self.downsample_channels[0],kernel_size=3,stride=1,padding = 1)

    # lets see the encoder
    self.encoder_down = nn.ModuleList([])
    for i in range(len(self.downsample_channels)-1):
      self.encoder_down.append(Encoder_Down(in_channel=self.downsample_channels[i],out_channel=self.downsample_channels[i+1],
                                            channel_group=self.group_norm_channels,n_layers=self.num_down_layers))
    self.encoder_mid = nn.ModuleList([])
    for i in range(len(self.mid_channels) - 1):
      self.encoder_mid.append(MidBlock(self.mid_channels[i],self.mid_channels[i+1],self.group_norm_channels,
                                       self.mid_num_layers,self.num_heads))
    self.encoder_norm = nn.GroupNorm(self.group_norm_channels, self.mid_channels[-1])
    self.encoder_conv_out = nn.Conv2d(self.mid_channels[-1], self.z_channels,kernel_size=3,padding = 1)

    # lets apply the quantization
    self.conv_before_quantization = nn.Conv2d(self.z_channels,self.z_channels,kernel_size=1)
    #codebook
    self.embedding = nn.Embedding(self.codebook_size, self.z_channels)

    #Decoder starts here
    self.conv_after_quantization = nn.Conv2d(self.z_channels,self.z_channels,kernel_size=1)
    self.decoder_conv_in = nn.Conv2d(self.z_channels,self.mid_channels[-1],kernel_size=3,padding = 1)

    self.decoder_mid = nn.ModuleList([])
    for i in reversed(range(1,len(self.mid_channels))):
      self.decoder_mid.append(MidBlock(self.mid_channels[i],self.mid_channels[i-1],self.group_norm_channels,self.mid_num_layers,
                                       self.num_heads))
    self.decoder_ups = nn.ModuleList([])
    for i in reversed(range(1,len(self.downsample_channels))):
      self.decoder_ups.append(UpBlock(self.downsample_channels[i],self.downsample_channels[i-1],self.group_norm_channels,
                                      self.num_up_layers))
    self.decoder_norm = nn.GroupNorm(self.group_norm_channels,self.downsample_channels[0])
    self.decoder_final_conv = nn.Conv2d(self.downsample_channels[0],self.image_channels,kernel_size=3,padding = 1)

  def encode(self,x):

    out = self.before_encoder(x)
    for down in self.encoder_down:
      out = down(out)
    for mid in self.encoder_mid:
      out = mid(out)

    out = self.encoder_norm(out)
    act = nn.SiLU()
    out = act(out)
    out = self.encoder_conv_out(out)
    out = self.conv_before_quantization(out)
    # print('Conv before Quantization', out.shape)

    b,c,h,w = out.shape
    out = out.permute(0,2,3,1) # b,h,w,c
    out = out.reshape(b,h*w,c) # channel wise vector finding the distance
    dist = torch.cdist(out,self.embedding.weight[None,:].repeat(out.size(0),1,1))
    # row wise see the min index it will become [B,h*w]
    min_dist_indices = torch.argmin(dist, dim = -1)
    # now pick the codebook i will get[b*h*w,c]
    quantization_out = torch.index_select(self.embedding.weight,dim = 0,index=min_dist_indices.view(-1))
    out = out.reshape((-1,out.size(-1))) # [b, h*w,c] --> [b*h*w,c]
    # make sure that encoder vector committed to one codebook vector
    commitment_loss = torch.mean((quantization_out.detach() - out)**2)

    #code book loss make sure that quantized vector close to encoded vector
    codebook_loss = torch.mean((quantization_out - out.detach()) ** 2)

    # straight through estimator so the gradients flow till encoder
    # detaching from the computation graph
    quantization_out = out + (quantization_out - out).detach()
    quantization_out = quantization_out.reshape((b,h,w,c)).permute(0,3,1,2)

    return quantization_out, commitment_loss, codebook_loss

  def decode(self, out_1):
    out_1 = self.conv_after_quantization(out_1)
    out_1 = self.decoder_conv_in(out_1)
    for mid in self.decoder_mid:
      out_1 = mid(out_1)
    for up in self.decoder_ups:
      out_1 = up(out_1)
    out_1 = self.decoder_norm(out_1)
    out_1 = self.decoder_final_conv(out_1)
    return out_1

  def forward(self,x):
    quantization_out,commitment_loss,codebook_loss = self.encode(x)
    # Now Upsampling with decoder start here
    out_1 = self.decode(quantization_out)
    return out_1, quantization_out, commitment_loss, codebook_loss

**PATCH GAN**

In [ ]:
# patch GAN Discriminator
# each and every pixel of output feature map points to a patch in original image
# for real image all the pixel value of output should be 1
# for generated image all the pixel value of output should be 0
class Discriminator(nn.Module):
  def __init__(self, image_channels):
    super().__init__()
    self.conv1 = nn.Sequential(
        nn.Conv2d(image_channels,64,kernel_size=4,stride=2,padding=1,bias = True),
        nn.LeakyReLU(0.2)
    )
    self.conv2 = nn.Sequential(
        nn.Conv2d(64,128,kernel_size = 4,stride = 2,padding = 1,bias = False),
        nn.BatchNorm2d(128),
        nn.LeakyReLU(0.2)
    )
    self.conv3 = nn.Sequential(
        nn.Conv2d(128,256,kernel_size = 4,stride = 2,padding = 1,bias = False),
        nn.BatchNorm2d(256),
        nn.LeakyReLU(0.2)
    )
    self.conv4 = nn.Sequential(
        nn.Conv2d(256,1,kernel_size = 4,stride = 2,padding = 1,bias = True),
        nn.LeakyReLU(0.2)
    )
  def forward(self,x):
    out = self.conv1(x)
    out = self.conv2(out)
    out = self.conv3(out)
    out = self.conv4(out)
    return out

# **Dataset Class**

In [ ]:
## dataset class
class ImageDataset(Dataset):
    def __init__(self, root_dir, im_size = 256):
        self.root_dir = root_dir
        self.image_paths = [
            os.path.join(root_dir, fname)
            for fname in os.listdir(root_dir)
            if fname.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]
        self.transform = transforms.Compose([
            transforms.Resize((im_size, im_size)),
            torchvision.transforms.CenterCrop(im_size),
            transforms.ToTensor(),           # [0,1]
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        im = Image.open(img_path).convert("RGB")
        im_tensor = self.transform(im)
        im_tensor = (2 * im_tensor) - 1
        im.close()
        return im_tensor

In [ ]:
!pip install lpips

# **Training Loop of LDM**

In [ ]:
import lpips
device = 'cuda' if torch.cuda.is_available() else 'cpu'
lpips_model = lpips.LPIPS(net='vgg').to(device)
lpips_model.eval()

for p in lpips_model.parameters():
    p.requires_grad = False

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: /root/miniconda3/envs/py3.10/lib/python3.10/site-packages/lpips/weights/v0.1/vgg.pth


In [ ]:
image_channels = 3
image_size = 256
dataset_path = 'jl_fs/celeba_hq_256'
learning_rate = 1e-5                  # autoencoder_lr
num_epochs = 20                       # autoencoder_epochs
batch_size = 4                        # autoencoder_batch_size
beta_1 = 0.5
beta_2 = 0.999
discriminator_step_start = 15000      # disc_start
acc_steps = 4                         # autoencoder_acc_steps
image_save_steps = 64                 # autoencoder_img_save_steps

BASE_DIR = "jl_fs/celebhq"                  # task_name
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(f"{BASE_DIR}/outputs/images", exist_ok=True)
os.makedirs(f"{BASE_DIR}/outputs/quant", exist_ok=True)
os.makedirs(f"{BASE_DIR}/model_store", exist_ok=True)

In [ ]:
codebook_weight = 1
commitment_beta = 0.2
disc_weight = 0.5
perceptual_weight = 0.1
disc_start = 8000
acc_steps = 4
step_count = 0
model = VQVAE().to(device)

discriminator = Discriminator(image_channels=image_channels).to(device)
celeb_dataset = ImageDataset(dataset_path, image_size)
celeb_dataloader = DataLoader(celeb_dataset, batch_size=batch_size, shuffle=True)

reconstruction_criterion = torch.nn.MSELoss()
discriminator_criterion = torch.nn.MSELoss()
optimizer_dis = optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(0.5, 0.999))
optimizer_gen = optim.Adam(model.parameters(), lr=learning_rate, betas=(0.5, 0.999))
step_count = 0

print(f'The length of celeb data loader is {len(celeb_dataloader)}')

for epoch in range(num_epochs):

    recon_losses = []
    codebook_losses = []
    perceptual_losses = []
    gen_losses = []
    disc_losses = []

    optimizer_gen.zero_grad()
    optimizer_dis.zero_grad()

    for batch in celeb_dataloader:

        batch = batch.float().to(device)
        step_count += 1


        recon, quantized, commit_loss, code_loss = model(batch)


        recon_loss = reconstruction_criterion(recon, batch)
        recon_losses.append(recon_loss.item())

        g_loss = (
            recon_loss
            + codebook_weight * code_loss
            + commitment_beta * commit_loss
        )

        codebook_losses.append(codebook_weight * code_loss.item())


        if step_count > disc_start:
            fake_pred = discriminator(recon)
            adv_loss = discriminator_criterion(
                fake_pred,
                torch.ones_like(fake_pred)
            )
            g_loss += disc_weight * adv_loss
            gen_losses.append(disc_weight * adv_loss.item())


        lpips_loss = torch.mean(lpips_model(recon, batch))
        g_loss += perceptual_weight * lpips_loss
        perceptual_losses.append(perceptual_weight * lpips_loss.item())


        g_loss = g_loss / acc_steps
        g_loss.backward()

        if step_count % acc_steps == 0:
            optimizer_gen.step()
            optimizer_gen.zero_grad()


        if step_count > disc_start:

            fake = recon.detach()

            fake_pred = discriminator(fake)
            real_pred = discriminator(batch)

            fake_loss = discriminator_criterion(
                fake_pred,
                torch.zeros_like(fake_pred)
            )
            real_loss = discriminator_criterion(
                real_pred,
                torch.ones_like(real_pred)
            )

            d_loss = disc_weight * (fake_loss + real_loss) / 2
            disc_losses.append(d_loss.item())

            d_loss = d_loss / acc_steps
            d_loss.backward()

            if step_count % acc_steps == 0:
                optimizer_dis.step()
                optimizer_dis.zero_grad()


        if step_count % 64 == 0 or step_count == 1:

            model.eval()
            with torch.no_grad():

                sample_size = min(8, batch.shape[0])
                real = batch[:sample_size]
                recon_img = recon[:sample_size]
                recon_img = torch.clamp(recon_img, -1., 1.)
                real = (real + 1) / 2
                recon_img = (recon_img + 1) / 2
                grid = torch.cat([real, recon_img], dim=0)
                vutils.save_image(
                    grid,
                    f"{BASE_DIR}/outputs/images/step_{step_count}.png",
                    nrow=sample_size
                )
            model.train()

    if len(disc_losses) > 0:
        print(
            f"Epoch {epoch+1} | Recon {np.mean(recon_losses):.4f} | "
            f"Perceptual {np.mean(perceptual_losses):.4f} | "
            f"Codebook {np.mean(codebook_losses):.4f} | "
            f"G {np.mean(gen_losses):.4f} | D {np.mean(disc_losses):.4f}"
        )
    else:
        print(
            f"Epoch {epoch+1} | Recon {np.mean(recon_losses):.4f} | "
            f"Perceptual {np.mean(perceptual_losses):.4f} | "
            f"Codebook {np.mean(codebook_losses):.4f}"
        )
torch.save({
    "model": model.state_dict(),
    "discriminator": discriminator.state_dict(),
}, f"{BASE_DIR}/model_store/vqvae_ckpt.pth")

print("Training Done")

The length of celeb data loader is 3886
Epoch 1 | Recon 0.0460 | Perceptual 0.0533 | Codebook 0.0042
Epoch 2 | Recon 0.0202 | Perceptual 0.0418 | Codebook 0.0034
Epoch 3 | Recon 0.0246 | Perceptual 0.0420 | Codebook 0.0026 | G 0.0293 | D 0.1869
Epoch 4 | Recon 0.0145 | Perceptual 0.0360 | Codebook 0.0027 | G 0.0310 | D 0.1605
Epoch 5 | Recon 0.0125 | Perceptual 0.0331 | Codebook 0.0029 | G 0.0313 | D 0.1582
Epoch 6 | Recon 0.0112 | Perceptual 0.0306 | Codebook 0.0031 | G 0.0314 | D 0.1574
Epoch 7 | Recon 0.0102 | Perceptual 0.0288 | Codebook 0.0030 | G 0.0315 | D 0.1570
Epoch 8 | Recon 0.0094 | Perceptual 0.0270 | Codebook 0.0031 | G 0.0315 | D 0.1567
Epoch 9 | Recon 0.0088 | Perceptual 0.0257 | Codebook 0.0030 | G 0.0315 | D 0.1565
Epoch 10 | Recon 0.0083 | Perceptual 0.0245 | Codebook 0.0030 | G 0.0316 | D 0.1563
Epoch 11 | Recon 0.0081 | Perceptual 0.0237 | Codebook 0.0030 | G 0.0317 | D 0.1560
Epoch 12 | Recon 0.0081 | Perceptual 0.0234 | Codebook 0.0029 | G 0.0317 | D 0.1560
Epoch

# **Once AutoEncoder Trained lets implement UNET for Diffusion model to denoise the image**

In [ ]:
class UNETDown(nn.Module):
  def __init__(self,in_channel,out_channel,t_emb_dim,num_heads,n_layers,group_norm_channels):
    super().__init__()
    self.t_emb_dim = t_emb_dim
    self.n_layers = n_layers
    self.residual = nn.ModuleList([
        ResidualBlock(in_channel if i == 0 else out_channel, out_channel,group_norm_channels, t_emb_info=self.t_emb_dim)
        for i in range(n_layers)
    ])
    self.mha = nn.ModuleList([
        MHAttention(group_norm_channels, out_channel,num_heads)
        for i in range(n_layers)
    ])
    self.downsample = nn.Conv2d(in_channels=out_channel, out_channels= out_channel,kernel_size=4,stride=2,padding=1)
  def forward(self,x,t_emb):
    out = x
    for i in range(self.n_layers):
      out = self.residual[i](out,t_emb)
      out = self.mha[i](out)
    out = self.downsample(out)
    return out

In [ ]:
class UNETMid(nn.Module):
  def __init__(self,in_channel, out_channel, channel_group, n_layers, n_heads, t_emb_dim):
    super().__init__()
    self.n_layers = n_layers
    self.residual_first = nn.ModuleList([
        ResidualBlock(in_channel if i == 0 else out_channel, out_channel,channel_group,t_emb_info=t_emb_dim)
        for i in range(n_layers + 1)
    ])
    self.mha = nn.ModuleList([
        MHAttention(channel_group, out_channel, n_heads)
        for i in range(n_layers)
    ])
  def forward(self, x,t_emb):
    out = x
    resnet_input = out
    out = self.residual_first[0](out,t_emb)
    for i in range(self.n_layers):
      out = self.mha[i](out)
      resnet_input = out
      out = self.residual_first[i+1](out,t_emb)
    return out

In [ ]:
class UNETUp(nn.Module):
  def __init__(self,in_channel, out_channel, channel_group,n_layers,n_heads,t_emb_dim):
    super().__init__()
    self.n_layers = n_layers
    self.residual = nn.ModuleList([
        ResidualBlock(in_channel if i == 0 else out_channel, out_channel,channel_group,t_emb_dim)
        for i in range(n_layers)
    ])
    self.upsample_conv = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(in_channel//2, in_channel//2, 3, padding=1),
            nn.GroupNorm(channel_group, in_channel//2),
            nn.SiLU()
        )
    self.mha = nn.ModuleList([
        MHAttention(channel_group, out_channel,n_heads)
        for i in range(n_layers)
    ])
  def forward(self,x,down_out, t_emb):
    x = self.upsample_conv(x)
    x = torch.cat([x,down_out],dim=1)
    for i in range(self.n_layers):
      x = self.residual[i](x,t_emb)
      x = self.mha[i](x)
    return x

In [ ]:
def sin_cos_time_embedding(t,t_emb_dim):
    # IN Attention all you need paper they done sin,cos alternatively
    # but at the end of the day the timestep vector need to be unique
    factor = 10000 ** ((torch.arange(
        start=0, end=t_emb_dim // 2, dtype=torch.float32, device=t.device) / (t_emb_dim // 2))
    )
    t_emb = t[:, None].repeat(1, t_emb_dim // 2) / factor
    # [b,256] , [b,256] row wise concat  [b,512]
    t_emb = torch.cat([torch.sin(t_emb), torch.cos(t_emb)], dim=-1)
    return t_emb

In [ ]:
class UNET(nn.Module):
    def __init__(self,image_channels = 3):
      super().__init__()
      self.down_channels = [256,384,512,768]
      self.mid_channels = [768,512]
      self.t_emb_dim = 512
      self.num_down_layers = 2
      self.num_mid_layers = 2
      self.num_up_layers = 2
      self.group_norm_channels = 32
      self.num_head = 16
      self.conv_out_channels = 128

      self.time_proj = nn.Sequential(
          nn.Linear(self.t_emb_dim,self.t_emb_dim),
          nn.SiLU(),
          nn.Linear(self.t_emb_dim,self.t_emb_dim)
      )

      self.conv_in = nn.Conv2d(image_channels, self.down_channels[0],kernel_size=3,padding=1)

      self.downs = nn.ModuleList([])
      for i in range(len(self.down_channels)-1):
        self.downs.append(UNETDown(self.down_channels[i],self.down_channels[i+1],self.t_emb_dim,self.num_head,self.num_down_layers,self.group_norm_channels))


      self.mids = nn.ModuleList([])
      for i in range(len(self.mid_channels)-1):
        self.mids.append(UNETMid(self.mid_channels[i],self.mid_channels[i+1],self.group_norm_channels,self.num_mid_layers,self.num_head,self.t_emb_dim))

      self.ups = nn.ModuleList([])
      for i in reversed(range(len(self.down_channels)-1)):
        # here we concatenate so multiply by 2
        self.ups.append(UNETUp(self.down_channels[i] * 2 ,self.down_channels[i-1] if i!=0 else self.conv_out_channels,self.group_norm_channels,self.num_up_layers,self.num_head,self.t_emb_dim))

      self.final_group_norm = nn.GroupNorm(self.group_norm_channels,self.conv_out_channels)
      self.final_conv_out = nn.Conv2d(self.conv_out_channels,image_channels,kernel_size=3,padding = 1)


    def forward(self, x, t):
      # this t shape is (b,) batch of random time steps is taken
      out = self.conv_in(x) #[b,256,h,w]
      t_emb = sin_cos_time_embedding(torch.as_tensor(t).long(), self.t_emb_dim)
      t_emb = self.time_proj(t_emb)
      down_output = []
      for down in self.downs:
        down_output.append(out)
        #[[b,256,h,w],[b,384,h/2,w/2],[b,512,h/4,w/4]]
        out = down(out,t_emb)

      #[b,768,h/8,w/8] is passed to the Midblock
      # first passed to the resnet of mid block [b,512,h/8,h/8]
      # self_attention + resnet, self_attention+resnet [b,512,h/8,h/8]
      for mid in self.mids:
        out = mid(out,t_emb)

      for up in self.ups:
        down_result = down_output.pop()
        # first upsample it then concat then pass to 2 layers of [resnet,selfAttention]
        out = up(out,down_result,t_emb)

      out = self.final_group_norm(out)
      act = nn.SiLU()
      out = act(out)
      out = self.final_conv_out(out)
      return out

# Forward Process of Latent DIffusion model

In [ ]:
class LinearNoiseScheduler:

    def __init__(self, num_timesteps, beta_start, beta_end):
        self.num_timesteps = num_timesteps
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.betas = torch.linspace(beta_start, beta_end, num_timesteps)
        self.alphas = 1. - self.betas
        self.alpha_cum_prod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alpha_cum_prod = torch.sqrt(self.alpha_cum_prod)
        self.sqrt_one_minus_alpha_cum_prod = torch.sqrt(1 - self.alpha_cum_prod)

    def add_noise(self, original, noise, t):
        original_shape = original.shape
        batch_size = original_shape[0]

        sqrt_alpha_cum_prod = self.sqrt_alpha_cum_prod.to(original.device)[t].reshape(batch_size)
        sqrt_one_minus_alpha_cum_prod = self.sqrt_one_minus_alpha_cum_prod.to(original.device)[t].reshape(batch_size)

        # Reshape till (B,) becomes (B,1,1,1) if image is (B,C,H,W)
        for _ in range(len(original_shape) - 1):
            sqrt_alpha_cum_prod = sqrt_alpha_cum_prod.unsqueeze(-1)
        for _ in range(len(original_shape) - 1):
            sqrt_one_minus_alpha_cum_prod = sqrt_one_minus_alpha_cum_prod.unsqueeze(-1)

        # Apply and Return Forward process equation
        return (sqrt_alpha_cum_prod.to(original.device) * original
                + sqrt_one_minus_alpha_cum_prod.to(original.device) * noise)

    def sample_prev_timestep(self, xt, noise_pred, t):
        x0 = ((xt - (self.sqrt_one_minus_alpha_cum_prod.to(xt.device)[t] * noise_pred)) /
              torch.sqrt(self.alpha_cum_prod.to(xt.device)[t]))
        x0 = torch.clamp(x0, -1., 1.)

        mean = xt - ((self.betas.to(xt.device)[t]) * noise_pred) / (self.sqrt_one_minus_alpha_cum_prod.to(xt.device)[t])
        mean = mean / torch.sqrt(self.alphas.to(xt.device)[t])

        if t == 0:
            return mean, x0
        else:
            variance = (1 - self.alpha_cum_prod.to(xt.device)[t - 1]) / (1.0 - self.alpha_cum_prod.to(xt.device)[t])
            variance = variance * self.betas.to(xt.device)[t]
            sigma = variance ** 0.5
            z = torch.randn(xt.shape).to(xt.device)
            return mean + sigma * z, x0
    #DDIM SAMPLING
    @torch.no_grad()
    def sample_prev_timestep_ddim(self,xt,noise_pred,t,t_prev,eta=0.0):
        if isinstance(t, torch.Tensor):
            t = t.item()
        if isinstance(t_prev, torch.Tensor):
            t_prev = t_prev.item()
        alpha_cumprod_t = self.alpha_cum_prod.to(xt.device)[t]

        if t_prev >= 0:
            alpha_cumprod_prev = self.alpha_cum_prod.to(xt.device)[t_prev]
        else:
            alpha_cumprod_prev = torch.tensor(1.0).to(xt.device)

        sqrt_alpha_t = torch.sqrt(alpha_cumprod_t)
        sqrt_one_minus_alpha_t = torch.sqrt(1 - alpha_cumprod_t)

        # Predict x0
        x0_pred = (
            xt - sqrt_one_minus_alpha_t * noise_pred
        ) / sqrt_alpha_t

        x0_pred = torch.clamp(x0_pred, -1., 1.)

        # DDIM sigma
        sigma = eta * (
            torch.sqrt((1 - alpha_cumprod_prev) / (1 - alpha_cumprod_t))
            * torch.sqrt(
                1 - alpha_cumprod_t / alpha_cumprod_prev
            )
        )

        noise = torch.randn_like(xt) if eta > 0 else 0

        # Direction pointing to x_t
        pred_dir = torch.sqrt(
            1 - alpha_cumprod_prev - sigma**2
        ) * noise_pred

        xt_prev = (
            torch.sqrt(alpha_cumprod_prev) * x0_pred
            + pred_dir
            + sigma * noise
        )

        return xt_prev, x0_pred

# TRAINING LATENT DIFFUSION MODEL

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
num_timesteps = 1000
batch_size = 64
num_epochs = 100
lr = 1e-4
image_size = 256
image_channels = 3
z_channels = 3


BASE_DIR = "jl_fs/celebhq"
vqvae_ckpt_path = f"{BASE_DIR}/model_store/vqvae_ckpt.pth"
ldm_ckpt_path = f"{BASE_DIR}/model_store/ddpm_faces_cosine_ckpt.pth"
vqvae = VQVAE().to(device)

ckpt = torch.load(vqvae_ckpt_path, map_location=device)
vqvae.load_state_dict(ckpt["model"])
vqvae.eval()

# Freeze VAE
for p in vqvae.parameters():
    p.requires_grad = False

model = UNET(z_channels).to(device)
model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = torch.nn.MSELoss()
scheduler = CosineScheduler().to(device)
dataset_path = 'jl_fs/celeba_hq_256'

celeb_dataset = ImageDataset(dataset_path, image_size)
celeb_dataloader = DataLoader(celeb_dataset, batch_size=batch_size, shuffle=True)

for epoch in range(num_epochs):
    losses = []
    for images in tqdm(celeb_dataloader):
        optimizer.zero_grad()
        images = images.float().to(device)

        with torch.no_grad():
            z,_,_ = vqvae.encode(images)   # [B, 3, H/8, W/8]
        noise = torch.randn_like(z)
        t = torch.randint(0,num_timesteps,(z.shape[0],),device=device).long()
        z_noisy = scheduler.add_noise(z, noise, t)
        noise_pred = model(z_noisy, t)
        loss = criterion(noise_pred, noise)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    torch.save({
        "model": model.state_dict(),
        'optimizer':optimizer.state_dict()
    }, ldm_ckpt_path)
    print(f"Epoch {epoch+1} completed | Avg Loss: {np.mean(losses):.4f}")

100%|██████████| 243/243 [04:08<00:00,  1.02s/it]


Epoch 1 completed | Avg Loss: 0.3859


100%|██████████| 243/243 [04:03<00:00,  1.00s/it]


Epoch 2 completed | Avg Loss: 0.3107


100%|██████████| 243/243 [03:59<00:00,  1.02it/s]


...

Epoch 74 completed | Avg Loss: 0.2401


100%|██████████| 243/243 [04:00<00:00,  1.01it/s]


Epoch 75 completed | Avg Loss: 0.2429


 16%|█▌        | 38/243 [00:38<03:27,  1.01s/it]


KeyboardInterrupt: 

In [ ]:
import torch
import torchvision.utils as vutils
import imageio
import os
from tqdm import tqdm
import cv2

@torch.no_grad()
def sample_ldm_gif(unet_model, vae, noise_scheduler,
                   num_timesteps, device, save_path):
    unet_model.eval()
    vqvae.eval()
    num_samples = 8

    xt_list = [torch.randn((1, 3, 32, 32)).to(device) for _ in range(num_samples)]
    frames = []
    ddim_steps = 100

    step_indices = torch.linspace(
        num_timesteps - 1,
        0,
        ddim_steps
    ).long()


    for t in tqdm(reversed(range(num_timesteps))):
        imgs_this_step = []

        for i in range(num_samples):
            xt = xt_list[i]

            t_tensor = torch.full((1,), t, device=device, dtype=torch.long)


            noise_pred = unet_model(xt, t_tensor)


            xt, _ = noise_scheduler.sample_prev_timestep(
                xt, noise_pred, t_tensor
            )

            xt_list[i] = xt


            decoded = vqvae.decode(xt)


            img = (decoded.clamp(-1, 1) + 1) / 2
            img = img.cpu()

            imgs_this_step.append(img)


        if t % 20 == 0 or t == num_timesteps - 1:

            imgs_this_step = torch.cat(imgs_this_step, dim=0)


            grid = vutils.make_grid(imgs_this_step, nrow=4)

            ndarr = (grid.permute(1, 2, 0).numpy() * 255).astype("uint8")


            ndarr = cv2.resize(
                ndarr,
                (256 * 4, 256 * 2),
                interpolation=cv2.INTER_NEAREST
            )

            frames.append(ndarr)


    os.makedirs(save_path, exist_ok=True)
    gif_path = os.path.join(save_path, "dit_faces_256_10.gif")

    imageio.mimsave(gif_path, frames, format='GIF', duration=0.15)

    print(f"GIF saved at: {gif_path}")
BASE_DIR = "jl_fs/celebhq"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
vqvae_ckpt_path = f"{BASE_DIR}/model_store/vqvae_ckpt.pth"
vqvae = VQVAE().to(device)
ckpt = torch.load(vqvae_ckpt_path, map_location=device)
vqvae.load_state_dict(ckpt["model"])
vqvae.eval()
# Freeze VAE
for p in vqvae.parameters():
    p.requires_grad = False

# image_height = 32
# image_width = 32
image_channels = 3
# t_emb_dim = 768
# n_transformer_layers = 12
# n_heads_in_attention = 12
# hidden_size = 768
# each_head_dim = 64
# patch_width = 2
# patch_height = 2
# z_channels = 3
# dit_model = DiffusionTransformer(image_height,image_width, image_channels,t_emb_dim,n_transformer_layers,n_heads_in_attention,
#                              hidden_size,each_head_dim,patch_height,patch_width).to(device)
# dit_ckpt = torch.load('jl_fs/celebhq/model_store/dit_celeb_ckpt.pth',map_location=device)
# dit_model.load_state_dict(dit_ckpt['model'])
unet_model = UNET(image_channels).to(device)
unet_ckpt = torch.load('jl_fs/celebhq/model_store/ddpm_faces_cosine_ckpt.pth',map_location=device)
unet_model.load_state_dict(unet_ckpt['model'])
num_timesteps = 1000
beta_start =  1e-4
beta_end = 0.02
scheduler = LinearNoiseScheduler(num_timesteps,beta_start,beta_end)
save_path = 'jl_fs/celebhq/outputs_dit_faces/images'
sample_ldm_gif(dit_model, vqvae, scheduler,num_timesteps, device, save_path)

1000it [01:44,  9.54it/s]


GIF saved at: jl_fs/celebhq/outputs_dit_faces/images/dit_faces_256_10.gif


**DDIM**

In [ ]:
import torch
import torchvision.utils as vutils
import imageio
import os
from tqdm import tqdm
import cv2


@torch.no_grad()
def sample_ldm_gif_ddim(
    unet_model,
    vqvae,
    noise_scheduler,
    num_timesteps,
    device,
    save_path,
    ddim_steps=100,
    eta=0.0
):

    unet_model.eval()
    vqvae.eval()

    num_samples = 8

    xt_list = [
        torch.randn((1, 3, 32, 32)).to(device)
        for _ in range(num_samples)
    ]


    step_indices = torch.linspace(
        num_timesteps - 1,
        0,
        ddim_steps
    ).long()

    frames = []


    for i in tqdm(range(len(step_indices) - 1)):

        t = step_indices[i]
        t_prev = step_indices[i + 1]

        imgs_this_step = []

        for j in range(num_samples):

            xt = xt_list[j]

            t_tensor = torch.full(
                (1,),
                t,
                device=device,
                dtype=torch.long
            )

            # Predict noise
            noise_pred = unet_model(xt, t_tensor)

            # DDIM step
            xt, _ = noise_scheduler.sample_prev_timestep_ddim(
                xt=xt,
                noise_pred=noise_pred,
                t=t,
                t_prev=t_prev,
                eta=eta
            )

            xt_list[j] = xt

            # Decode latent
            decoded = vqvae.decode(xt)

            img = (decoded.clamp(-1, 1) + 1) / 2

            imgs_this_step.append(img.cpu())


        if i % 5 == 0 or i == len(step_indices) - 2:

            imgs_this_step = torch.cat(imgs_this_step, dim=0)

            grid = vutils.make_grid(
                imgs_this_step,
                nrow=4
            )

            ndarr = (
                grid.permute(1, 2, 0).numpy() * 255
            ).astype("uint8")


            ndarr = cv2.resize(
                ndarr,
                (128 * 4, 128 * 2),   # 512 x 256
                interpolation=cv2.INTER_AREA
            )

            frames.append(ndarr)


    os.makedirs(save_path, exist_ok=True)

    gif_path = os.path.join(
        save_path,
        f"ddim_faces_{ddim_steps}_steps_1.gif"
    )

    imageio.mimsave(
        gif_path,
        frames,
        format='GIF',
        duration=0.25   # slower fps = smaller size
    )

    print(f"\nGIF saved at: {gif_path}")

In [ ]:
BASE_DIR = "jl_fs/celebhq"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
vqvae_ckpt_path = f"{BASE_DIR}/model_store/vqvae_ckpt.pth"
vqvae = VQVAE().to(device)
ckpt = torch.load(vqvae_ckpt_path, map_location=device)
vqvae.load_state_dict(ckpt["model"])   # ✅ IMPORTANT
vqvae.eval()
# Freeze VAE
for p in vqvae.parameters():
    p.requires_grad = False

image_channels = 3
unet_model = UNET(image_channels).to(device)
unet_ckpt = torch.load('jl_fs/celebhq/model_store/ddpm_faces_ckpt.pth',map_location=device)
unet_model.load_state_dict(unet_ckpt['model'])
num_timesteps = 1000
beta_start =  1e-4
beta_end = 0.02
scheduler = LinearNoiseScheduler(num_timesteps,beta_start,beta_end)
save_path = 'jl_fs/celebhq/outputs_ddim_faces/images'

In [ ]:
sample_ldm_gif_ddim(
    unet_model=unet_model,   # or UNET model
    vqvae=vqvae,
    noise_scheduler=scheduler,
    num_timesteps=1000,
    device=device,
    save_path=save_path,
    ddim_steps=100,   # try 50 / 100 / 250
    eta=0.0
)

100%|██████████| 99/99 [00:15<00:00,  6.23it/s]



GIF saved at: jl_fs/celebhq/outputs_ddim_faces/images/ddim_faces_100_steps_1.gif


In [ ]:
pip install imageio

Note: you may need to restart the kernel to use updated packages.


In [ ]:
!pip install opencv-python

  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl (72.9 MB)


98439427 for UNET FACES

In [ ]:
sum(p.numel() for p in unet_model.parameters())

98439427

In [ ]:
!pip install einops

# DIT[DIFFUSION TRANSFORMER]

In [ ]:
from einops import rearrange ## used for chaning the dimension
class PatchEmbedding(nn.Module):
  def __init__(self, image_height, image_width, image_channels,patch_height, patch_width, hidden_size):
    super().__init__()
    self.image_height = image_height
    self.image_width = image_width
    self.image_channels = image_channels
    self.patch_height = patch_height
    self.patch_width = patch_width
    self.hidden_size = hidden_size
    each_patch_dim = self.patch_height * self.patch_width * self.image_channels
    self.patch_embed = nn.Linear(each_patch_dim, self.hidden_size)

    ## Different Weights used in paper
    nn.init.xavier_uniform_(self.patch_embed.weight)
    nn.init.constant_(self.patch_embed.bias, val = 0)


  def get_position_embedding(self,hidden_size,num_patches_along_height,num_patches_along_width,device):
    # get the position embedding of patch
    grid_h = torch.arange(num_patches_along_height,dtype = torch.float32, device=device)
    grid_w = torch.arange(num_patches_along_width,dtype=torch.float32,device= device)
    grid = torch.meshgrid(grid_h,grid_w,indexing='ij')
    grid = torch.stack(grid, dim=0) # one grid above another
    g1 = grid[0].reshape(-1) #1d tensor
    g2 = grid[1].reshape(-1) #1d tensor

    # applying the 2d position embedding
    # total 768 dim --> (384 along height) and (384 along width)
    # along height 384 dim in that 192 dim for sin and 192 dim for cos
    # similarly for the width also
    factor = 10000 ** (torch.arange(0, hidden_size//4,dtype=torch.float32,device = device)/(hidden_size//4))
    grid_h_emb = (g1[:,None].repeat(1,hidden_size//4))/factor #(num_patches_along_height, 192)
    grid_w_emb = (g2[:,None].repeat(1,hidden_size//4))/factor #(num_patches_along_width, 192)

    grid_h_emb = torch.cat([torch.sin(grid_h_emb),torch.cos(grid_h_emb)],dim=-1) # 192 + 192 = 384 along height
    grid_w_emb = torch.cat([torch.sin(grid_w_emb),torch.cos(grid_w_emb)],dim=-1) # 192 + 192 = 384 along width
    # 384 + 384 = 768
    pos_emb = torch.cat([grid_h_emb,grid_w_emb],dim = -1)
    return pos_emb


  def forward(self,x):
    num_patches_along_height = self.image_height // self.patch_height
    num_patches_along_width = self.image_width // self.patch_width

    # height of image = num_patches_along_height * height of patch
    #[batch_size , channels, num_patches_along_height * patch_height, num_patches_along_width*patch_width]
    # convert to like this.
    #[batch_size,num_patches_along_height * num_patches_along_width, patch_width*patch_height*channels]
    out = rearrange(x, pattern = 'b c (nh ph) (nw pw) -> b (nh nw) (ph pw c)', ph = self.patch_height, pw=self.patch_width)

    #(ph pw c) is the each_patch_dim
    out = self.patch_embed(out) # [b, (nh ph), (ph pw c)] --> [b, (nh,ph) , self.hidden_size]
    # definetly broadcasting is applied here
    out = out + self.get_position_embedding(self.hidden_size, num_patches_along_height, num_patches_along_width, device = x.device)
    return out


In [ ]:
class Attention(nn.Module):
  # hidden_size = 768 and num_heads = 12, head_dim = 64
  def __init__(self, hidden_size, num_heads, head_dim):
    super().__init__()

    self.hidden_size = hidden_size
    self.num_heads = num_heads
    self.head_dim = head_dim

    self.attn_dim = self.num_heads * self.head_dim
    self.qkv = nn.Linear(self.hidden_size, 3*self.attn_dim)

    self.output_proj = nn.Sequential(
            nn.Linear(self.attn_dim, self.hidden_size))

    nn.init.xavier_uniform_(self.qkv.weight)
    nn.init.constant_(self.qkv.bias, 0)
    nn.init.xavier_uniform_(self.output_proj[0].weight)
    nn.init.constant_(self.output_proj[0].bias, 0)

  def forward(self,x):
    # self.attn_dim = 12 * 64 = 768
    #[b, num_patches_along_height * num_patches_along_width, 768] --> [b, num_patches_along_height * num_patches_along_width, 3*self.attn_dim]
    q,k,v = self.qkv(x).split(self.attn_dim,dim=-1)

    # num_patches = num_patches_along_height * num_patches_along_width
    # q = [B, num_patches, attn_dim] --> [B, num_patches, num_heads * head_dim]
    # k = [B, num_patches, attn_dim] --> [B, num_patches, num_heads * head_dim]
    # v = [B, num_patches, attn_dim] --> [B, num_patches, num_heads * head_dim]
    q = rearrange(q,'b n (n_h h_dim) -> b n_h n h_dim',n_h = self.num_heads,h_dim=self.head_dim)
    k = rearrange(k,'b n (n_h h_dim) -> b n_h n h_dim',n_h = self.num_heads,h_dim=self.head_dim)
    v = rearrange(v,'b n (n_h h_dim) -> b n_h n h_dim',n_h = self.num_heads,h_dim=self.head_dim)
    #[B, n_h , n,h_dim] [B,n_h,h_dim,n] --> [B,n_h,n,n]
    q_k_T = torch.matmul(q , k.transpose(2,3)) * (self.head_dim ** (-0.5))
    q_k_T = F.softmax(q_k_T,dim=-1)

    #[B,n_h,n,n] *[B,n_h,n,h_dim] --> [B,n_h,n,h_dim]
    out = torch.matmul(q_k_T,v)
    out = rearrange(out, 'b n_h n h_dim  -> b n (n_h h_dim)')

    out = self.output_proj(out)

    #[b,n,768]
    return out

In [ ]:
class MyTransformer(nn.Module):
  # archiytecture
  #Layer_norm ---> Multihead attention ---> residual ----> Layer_norm---> MLP----> residual
  # for Layer Norm we use Adapative Layer norm [means using outer scale and shift parameters]
  def __init__(self,hidden_size,num_heads_in_attention,each_head_dim):
    super().__init__()
    self.hidden_size = hidden_size
    self.num_heads_in_attention = num_heads_in_attention
    self.each_head_dim = each_head_dim
    # used in MLP of transformer
    self.ffn = 4 * self.hidden_size

    # element wise affine we use external scale and shift just x-mu/sigma is done, eps denominator not to become zero
    self.layer_norm_before_attn = nn.LayerNorm(self.hidden_size,elementwise_affine=False, eps =1e-6)
    self.attention = Attention(self.hidden_size, self.num_heads_in_attention, self.each_head_dim)

    self.layer_norm_before_mlp = nn.LayerNorm(self.hidden_size,elementwise_affine=False, eps =1e-6)
    self.mlp = nn.Sequential(
                nn.Linear(self.hidden_size, self.ffn),
                nn.GELU(approximate='tanh'), # gelu is x * phi(x) where phi(x) is cummulative distribution of gaussian with mean = 0 and std = 1,
                nn.Linear(self.ffn,self.hidden_size)
    )
    # time embedding pass into this mlp and get gamma and beta
    # gamma_1, beta_1 and gamma_2, beta_2, alpha_1 and alpha_2 is multiplied before residual
    # SiLU() is x * sigmoid(x)
    self.adaptive_norm_mlp = nn.Sequential(
        nn.SiLU(),
        nn.Linear(self.hidden_size, 6 * self.hidden_size)
    )

    # PAPER USED DIFFERENT WEIGHTS
    # transformer mlp
    nn.init.xavier_uniform_(self.mlp[0].weight)
    nn.init.constant_(self.mlp[0].bias, val=0)
    nn.init.xavier_uniform_(self.mlp[-1].weight)
    nn.init.constant_(self.mlp[-1].bias, val=0)

    #mlp of adaptive Layer Norm
    nn.init.constant_(self.adaptive_norm_mlp[-1].weight, val=0)
    nn.init.constant_(self.adaptive_norm_mlp[-1].bias, val=0)

  def forward(self,x,t_emb):
    # Initially DIT become Identity block at start

    #total_tokens = num_patches_along_width * num_patches_along_height
    #each token 768
    # shape of x is [B,total_token, 768]
    # t_emb is passed through the sin cos and i get the 784 dim
    scale_shift_aplha_param = self.adaptive_norm_mlp(t_emb) #[b, 6*self.hidden_size]
    scale_shift_aplha_param = scale_shift_aplha_param.chunk(6, dim = 1)
    # each is [B, self.hidden_size] = [B, 768]
    shift_before_attn, scale_before_attn, shift_before_mlp,scale_before_mlp,alpha_1,alpha_2 = scale_shift_aplha_param


    # x is after adding position embedding + token embedding
    out = x
    #[b,768] --> unsqueeze(1) =----> [b,1,768]
    # adding 1 because we set the self.adaptive_norm_mlp weight and bias as 0
    # if 1 is not added then my input become 0
    out = self.layer_norm_before_attn(x) * (1 + scale_before_attn.unsqueeze(1))  + shift_before_attn.unsqueeze(1)
    out = self.attention(out)
    out = alpha_1.unsqueeze(1) * out

    out = x + out

    y = out
    out = self.layer_norm_before_mlp(out) * (1 + scale_before_mlp.unsqueeze(1)) + shift_before_mlp.unsqueeze(1)
    out = self.mlp(out)
    out = alpha_2.unsqueeze(1) * out

    out = y + out
    return out

In [ ]:
class DiffusionTransformer(nn.Module):
  def __init__(self, image_height, image_width, image_channels,t_emb_dim,n_transformer_layers,n_heads_in_attention, hidden_size,each_head_dim,patch_height,patch_width):
    super().__init__()
    self.image_height = image_height
    self.image_width = image_width
    self.image_channels = image_channels
    self.t_emb_dim = t_emb_dim
    self.hidden_size = hidden_size

    self.patch_height = patch_height
    self.patch_width = patch_width

    self.n_transformer_layers = n_transformer_layers
    self.n_heads_in_attention = n_heads_in_attention
    self.each_head_dim = each_head_dim

    self.num_patches_height = self.image_height // self.patch_height
    self.num_patches_width = self.image_width // self.patch_width

    self.patch_embed_layer = PatchEmbedding(self.image_height,self.image_width,self.image_channels,self.patch_height,self.patch_width,self.hidden_size)

    # time embedding of sin and cos pass through the initial mlp we seen this
    self.time_mlp = nn.Sequential(
        nn.Linear(self.t_emb_dim,self.hidden_size),
        nn.SiLU(),
         nn.Linear(self.hidden_size, self.hidden_size),
    )

    self.all_transfomer_blocks = nn.ModuleList([
        MyTransformer(self.hidden_size,self.n_heads_in_attention,self.each_head_dim) for _ in range(self.n_transformer_layers)
    ])


    # after all transformer blocks

    self.final_norm = nn.LayerNorm(self.hidden_size,elementwise_affine=False,eps = 1e-6)
    # at output also we are injecting the scale and shift from outside
    self.adaptive_final_norm = nn.Sequential(
                                nn.SiLU(),
                                nn.Linear(self.hidden_size, 2* self.hidden_size)
                                )
    # total_tokens = number_of_height_patches * number_of_width_patches
    #[B, total_tokens, hidden_size] ---> [B, total_tokens, self.patch_width*self.patch_height*self.image_channels]
    self.final_projection = nn.Sequential(
        nn.Linear(self.hidden_size, self.patch_width * self.patch_height * self.image_channels)
    )

    # mentioned in paper
    nn.init.normal_(self.time_mlp[0].weight, std = 0.02)
    nn.init.normal_(self.time_mlp[2].weight,std = 0.02)
    nn.init.constant_(self.final_projection[0].weight,val=0)
    nn.init.constant_(self.final_projection[0].bias,val=0)

  def forward(self,x,t):
    # x = [B,C,H,W]
    # t = (B,)

    #[B,C,H,W] --> [B, C, num_patches_along_height*patch_height, num_patches_along_width*patch_width]
    # [B, num_patches_along_height * num_patches_along_width, patch_width*patch_height*image_channels]
    # to [B, num_patches_along_height * num_patches_along_width, 768]
    out = self.patch_embed_layer(x)
    # print(f'Output after patch embedding {out.shape}')


    #[B, t_emb_dim] t_emb_dim = 768
    t_emb = sin_cos_time_embedding(torch.as_tensor(t).long(), self.t_emb_dim)
    # print(f'Output after sin cos embedding {t_emb.shape}')

    #[B, self.hidden_size] = [B,768]
    t_emb = self.time_mlp(t_emb)
    # print(f'Output after time_mlp {t_emb.shape}')

    #[B,num_patches_along_height * num_patches_along_width, 768]
    for layer in self.all_transfomer_blocks:
      out = layer(out,t_emb)
      # print(f'Output after Transformer {out.shape}')

    #[B,768] --> [B, 2*768]
    final_mlp_scale,final_mlp_shift = self.adaptive_final_norm(t_emb).chunk(2,dim=1)
    # print(f'final_mlp_scale {final_mlp_scale.shape} final_mlp_shift {final_mlp_shift.shape}')
    out = self.final_norm(out)
    # print(f'After Final norm {out.shape}')
    out = out * (1 + final_mlp_scale.unsqueeze(1))  + final_mlp_shift.unsqueeze(1)

    out = self.final_projection(out)
    #[B,tokens,ph*pw*channels] --> [B, C, H, w]
    # print(out.shape)
    out = rearrange(out, 'b (nh nw) (ph pw c) -> b c (nh ph) (nw pw)',pw=self.patch_width,
                    ph=self.patch_height, nh = self.num_patches_height,
                    nw=self.num_patches_width, c = self.image_channels)
    return out

In [ ]:
@torch.no_grad()
def sample_one_image(model, vqvae, scheduler, device, img_size=64, channels=3):
    model.eval()
    xt = torch.randn((1, channels, img_size, img_size)).to(device)
    for t in reversed(range(scheduler.num_timesteps)):
        t_tensor = torch.tensor([t], device=device, dtype=torch.long)
        noise_pred = model(xt, t_tensor)
        xt, x0 = scheduler.sample_prev_timestep(xt, noise_pred, t)
    xt = vqvae.decode(xt)
    final_img = (xt.clamp(-1, 1) + 1) / 2  # [-1,1] → [0,1]
    model.train()
    return final_img

In [ ]:
# CELEB FACES
BASE_DIR = "jl_fs/celebhq"
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(BASE_DIR + "/outputs_dit_faces/images", exist_ok=True)
os.makedirs(BASE_DIR + "/model_store", exist_ok=True)
os.makedirs(BASE_DIR + "/output_training/", exist_ok=True)
num_epochs = 200
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ldm_ckpt_path = f"{BASE_DIR}/model_store/dit_faces_ckpt.pth"

vqvae = VQVAE().to(device)

ckpt = torch.load('jl_fs/celebhq/model_store/vqvae_ckpt.pth', map_location=device)
vqvae.load_state_dict(ckpt["model"])   # ✅ IMPORTANT
vqvae.eval()

# Freeze VAE
for p in vqvae.parameters():
    p.requires_grad = False
num_timesteps = 1000
beta_start = 1e-4
beta_end = 2e-2
learning_rate = 1e-5
image_size = 256
image_height = 32
image_width = 32
image_channels = 3
t_emb_dim = 768
n_transformer_layers = 12
n_heads_in_attention = 12
hidden_size = 768
each_head_dim = 64
patch_width = 2
patch_height = 2
dataset_path = 'jl_fs/celeba_hq_256/'
noise_scheduler = LinearNoiseScheduler(num_timesteps, beta_start,beta_end)
celeb_dataset = ImageDataset(dataset_path,image_size)
celeb_dataloader = DataLoader(celeb_dataset, batch_size= 128, shuffle = True)

dit_model = DiffusionTransformer(image_height,image_width, image_channels,t_emb_dim,n_transformer_layers,n_heads_in_attention,
                             hidden_size,each_head_dim,patch_height,patch_width).to(device)
dit_model.train()
optimizer_dit = optim.Adam(dit_model.parameters(),lr=learning_rate)
loss_criterion = torch.nn.MSELoss()



for epoch in range(num_epochs):
  losses = []
  for batch in tqdm(celeb_dataloader):
    optimizer_dit.zero_grad()
    batch = batch.to(device)

    with torch.no_grad():
      z,_,_ = vqvae.encode(batch)

    # i got the noise same as (batch,channel,height,width)
    noise = torch.randn_like(z).to(device)
    # pick the random time step
    randn_timestep = torch.randint(0,1000,(z.shape[0],)).to(device)

    noisy_image = noise_scheduler.add_noise(z,noise,randn_timestep)
    # print('noisy image shape',noisy_image.shape)
    noise_pred = dit_model(noisy_image,randn_timestep)

    loss = loss_criterion(noise_pred,noise)
    losses.append(loss.item())
    loss.backward()
    optimizer_dit.step()
  if epoch % 5 == 0:
    img = sample_one_image(dit_model,vqvae,noise_scheduler, device,32,3)
    vutils.save_image(img, f"{BASE_DIR}/output_training/sample_epoch_{epoch}.png")
    torch.save({'model':dit_model.state_dict(),'optimizer':optimizer_dit.state_dict()
                 },f"{BASE_DIR}/model_store/dit_celeb_ckpt.pth")
  print(f'epoch {epoch + 1} loss: {np.mean(losses)}')

100%|██████████| 122/122 [03:37<00:00,  1.78s/it]


epoch 1 loss: 0.8186330946742512


100%|██████████| 122/122 [03:36<00:00,  1.78s/it]


epoch 2 loss: 0.3627107379133584


100%|██████████| 122/122 [03:37<00:00,  1.78s/it]


...

epoch 190 loss: 0.23069147728994244


100%|██████████| 122/122 [03:36<00:00,  1.78s/it]


epoch 191 loss: 0.23548598636369236


  6%|▌         | 7/122 [00:13<03:43,  1.94s/it]


KeyboardInterrupt: 

In [ ]:
# unet_model = UNET(image_channels=3).to(device)
dit_model.load_state_dict(torch.load(f"{BASE_DIR}/model_store/dit_celeb_ckpt.pth"))
dit_model.eval()

sample_ddpm(
    dit_model,
    noise_scheduler,
    num_timesteps=1000,
    device=device,
    save_path=f"{BASE_DIR}/outputs_faces_dit/images"
)

1000it [02:24,  6.93it/s]


GIF saved at: jl_fs/outputs_faces_dit/images/ditfaces_grid_128.gif


In [ ]:
@torch.no_grad()
def sample_one_image(model, vqvae, scheduler, device, img_size=64, channels=3):
    model.eval()
    xt = torch.randn((1, channels, img_size, img_size)).to(device)
    for t in reversed(range(scheduler.num_timesteps)):
        t_tensor = torch.tensor([t], device=device, dtype=torch.long)
        x = model(xt, t_tensor)
        noise_pred, v_mixing = torch.chunk(x,chunks=2,dim=1)
        xt, x0 = scheduler.sample_prev_timestep(xt, noise_pred,v_mixing, t)
    xt = vqvae.decode(xt)
    final_img = (xt.clamp(-1, 1) + 1) / 2  # [-1,1] → [0,1]
    model.train()
    return final_img